In [ ]:
import logging
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class FGState(TypedDict):
    input_name: Optional[str]
    name: Optional[str]
    is_capital: Optional[bool]
    capital_name: Optional[str]
    final_statement: Optional[str]

def getting_name(state: FGState) -> FGState:
    name = state.get("input_name") 
    if not name:
        raise ValueError("input_name must be provided")
    state["name"] = name
    logger.info(f"getting_name: State = {state}")
    return state

def check_capital(state: FGState) -> str:
    in_name = state.get("name")
    if not in_name:
        raise ValueError("name must be set before checking capital")
    next_node = "final_statement_node" if in_name.isupper() else "convert_capital_agent"
    logger.info(f"check_capital: Name = {in_name}, Next node = {next_node}, State = {state}")
    state["is_capital"] = False if not in_name.isupper() else True
    return next_node

def final_stat(state: FGState) -> FGState:
    final_name = state.get("capital_name", state["name"])
    if not final_name:
        raise ValueError("No name available for final statement")
    state["final_statement"] = f"hello welcome to the capital club, MR.{final_name}"
    logger.info(f"final_stat: State = {state}")
    return state

def convert_capital(state: FGState) -> FGState:
    name_tobe_capitalized = state.get("name")
    if not name_tobe_capitalized:
        raise ValueError("name must be set before capitalization")
    state["capital_name"] = name_tobe_capitalized.upper()
    logger.info(f"convert_capital: State = {state}")
    return state

# Initialize the workflow
workflow = StateGraph(FGState)
workflow.add_node("getting_name_agent", getting_name)
workflow.add_node("final_statement_node", final_stat)
workflow.add_node("convert_capital_agent", convert_capital)

workflow.set_entry_point("getting_name_agent")
workflow.add_conditional_edges(
    "getting_name_agent",
    check_capital,
    {
        "final_statement_node": "final_statement_node",
        "convert_capital_agent": "convert_capital_agent"
    }
)
workflow.add_edge("convert_capital_agent", "final_statement_node")
workflow.set_finish_point("final_statement_node")

# Compile the graph
graph_capital = workflow.compile()

In [ ]:
# Test the graph
initial_state = {"input_name": "JHON"}
result = graph_capital.invoke(initial_state)
print(result["final_statement"])

INFO:__main__:getting_name: State = {'input_name': 'JHON', 'name': 'JHON'}
INFO:__main__:check_capital: Name = JHON, Next node = final_statement_node, State = {'input_name': 'JHON', 'name': 'JHON'}
INFO:__main__:final_stat: State = {'input_name': 'JHON', 'name': 'JHON', 'final_statement': 'hello welcome to the capital club, MR.JHON'}


hello welcome to the capital club, MR.JHON
